# 1. Bibliotecas e dispositivo





In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

PyTorch: 2.11.0+cpu
Dispositivo: cpu


### Explicação

Esta célula importa o PyTorch e define onde os cálculos serão realizados.

* **torch** é a biblioteca principal.
* **torch.nn** fornece as classes usadas para construir redes neurais.
* **torch.nn.functional** fornece funções como softmax.

A variável `device` recebe:
* `cuda`, se houver uma GPU disponível;
* `cpu`, caso contrário.

Assim, posteriormente podemos colocar o modelo e os dados no mesmo dispositivo.

# 2. Texto utilizado para o treinamento

In [32]:
text = """
Era uma vez um gato que vivia em uma pequena casa.
O gato gostava de caminhar pelo jardim.
Todos os dias o gato observava os pássaros.
Quando o sol aparecia, o gato saía de casa.
O cachorro também vivia naquela casa.
O cachorro corria pelo jardim e brincava com o gato.
O gato gostava de peixe e o cachorro gostava de carne.
À noite, os dois animais voltavam para casa.
O gato dormia perto da janela e o cachorro dormia perto da
porta.
No dia seguinte, tudo começava novamente.
""" * 200

print("Número de caracteres:", len(text))
print()
print(text[:500])



Número de caracteres: 95800


Era uma vez um gato que vivia em uma pequena casa.
O gato gostava de caminhar pelo jardim.
Todos os dias o gato observava os pássaros.
Quando o sol aparecia, o gato saía de casa.
O cachorro também vivia naquela casa.
O cachorro corria pelo jardim e brincava com o gato.
O gato gostava de peixe e o cachorro gostava de carne.
À noite, os dois animais voltavam para casa.
O gato dormia perto da janela e o cachorro dormia perto da
porta.
No dia seguinte, tudo começava novamente.

Era uma vez um gato 


### Explicação

Aqui criamos o conjunto de dados textual. O texto é repetido 200 vezes com:

`""" ...""" * 200`

Isso aumenta a quantidade de exemplos disponíveis para treinamento. Depois:

`len(text)`

calcula o número total de caracteres. E:

`text[:500]`

mostra apenas os primeiros 500 caracteres. Neste projeto, trabalharemos inicialmente com tokens que são caracteres individuais.

# 3. Construção do vocabulário

In [ ]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
  return [stoi[ch] for ch in s]

def decode(ids):
  return ''.join(itos[i] for i in ids)

print("Vocabulário:")
print(chars)
print()
print("|V| =", vocab_size)

Vocabulário:
['\n', ' ', ',', '.', 'E', 'N', 'O', 'Q', 'T', 'a', 'b', 'c', 'd', 'e', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'z', 'À', 'á', 'ç', 'é', 'í']

|V| = 36


### Explicação

O modelo não trabalha diretamente com caracteres. Ele trabalha com números.
Primeiro construímos o vocabulário:

`chars = sorted(list(set(text)))`

`set(text)` elimina caracteres repetidos. Assim, `chars` contém cada caractere diferente existente no texto. Por exemplo, podemos ter caracteres como:

`' ', ',', '.', 'A', 'E', ... , 'a', 'b', 'c', ...`

`vocab_size` é o tamanho desse vocabulário:

**Visão matemática**
|V| = número de caracteres diferentes.

Criamos então dois dicionários:

`stoi`

significa string to integer:

**Visão matemática**
caractere → inteiro.

Já:

`itos`

significa integer to string:

**Visão matemática**
inteiro → caractere.

As funções `encode` e `decode` realizam essas duas conversões.

# 4. Testando a tokenização

In [ ]:
example = "gato"

encoded = encode(example)
decoded = decode(encoded)

print("Texto:       ", example)
print("Tokens:      ", encoded)
print("Reconstrução:", decoded)

Texto:        gato
Tokens:       [14, 9, 26, 21]
Reconstrução: gato


### Explicação

Aqui verificamos se a codificação e a decodificação funcionam. O texto:

`gato`

é transformado em uma sequência de números:

**Visão matemática**
gato → [i1, i2, i3, i4].

Depois fazemos o caminho inverso:

**Visão matemática**
[i1, i2, i3, i4] → gato.

Isso confirma que conseguimos passar do texto para a representação numérica e retornar ao texto original.

# 5. Transformando todo o texto em um tensor

In [ ]:
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print("Shape:", data.shape)
print("Primeiros 50 tokens:")
print(data[:50])

Shape: torch.Size([96000])
Primeiros 50 tokens:
tensor([ 0,  4, 24,  9,  1, 27, 19,  9,  1, 28, 13, 30,  1, 27, 19,  1, 14,  9,
        26, 21,  1, 23, 27, 13,  1, 28, 16, 28, 16,  9,  1, 13, 19,  1, 27, 19,
         9,  1, 22, 13, 23, 27, 13, 20,  9,  1, 11,  9, 25,  9])


### Explicação

Agora codificamos todo o texto. O resultado é um tensor unidimensional:

**Visão matemática**
data $\in \mathbb{N}^N$,

onde N é o número total de caracteres. Cada posição contém o índice de um caractere no vocabulário. Usamos:

`dtype = torch.long`

porque os índices utilizados por nn.Embedding precisam ser inteiros.

# 6. Separando treinamento e validação

In [ ]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Tokens de treino:       ", len(train_data))
print("Tokens de validação:    ", len(val_data))

Tokens de treino:        86400
Tokens de validação:     9600


### Explicação

Dividimos os dados em duas partes:

**Visão matemática**
90% → treinamento
e
10% → validação.

O modelo será treinado utilizando `train_data`. `val_data` é separado para podermos avaliar posteriormente o comportamento do modelo em dados que não foram usados diretamente para atualizar seus parâmetros.

# 7. Hiperparâmetros

In [ ]:
d = 32        # dimensão do embedding
T = 64        # tamanho máximo do contexto
L = 2         # número de blocos Transformer
h = 4         # número de cabeças de atenção

batch_size = 32

print("|V|  =", vocab_size)
print("d    =", d)
print("T    =", T)
print("L    =", L)
print("h    =", h)
print("batch_size =", batch_size)

|V|  = 36
d    = 32
T    = 64
L    = 2
h    = 4
batch_size = 32


### Explicação

Aqui definimos os principais hiperparâmetros do MiniGPT.

**Dimensão do embedding**
d = 32.

Cada caractere será representado por um vetor de dimensão 32.

**Tamanho do contexto**
T = 64.

O modelo poderá observar, no máximo, 64 caracteres simultaneamente.

**Número de blocos**
L = 2.

A arquitetura terá dois blocos Transformer.

**Número de cabeças**
h = 4.

Cada bloco de atenção terá quatro cabeças.

**Tamanho do batch**
B = 32.

Cada atualização utilizará 32 sequências simultaneamente.

# 8. Construção dos lotes

In [ ]:
def get_batch(split):
  source = train_data if split == "train" else val_data

  starts = torch.randint(
      0,
      len(source) - T - 1,
      (batch_size,)
  )

  X = torch.stack([
      source[i:i+T]
      for i in starts
  ])

  Y = torch.stack([
      source[i+1:i+T+1]
      for i in starts
  ])

  return X, Y

### Explicação

Essa função cria exemplos para o treinamento. A ideia fundamental de um modelo de linguagem é:

**Conceito-chave**:
prever o próximo token

Por isso, X e Y são deslocados uma posição. Se tivermos:

`O gato dorme`

podemos construir:

**Visão matemática**
X = O gato dorm
e
Y = gato dorme.

Formalmente:

**Visão matemática**
$Y_t = X_{t+1}$.

Cada posição de X é usada para prever a posição seguinte em Y.

# 9. Visualizando um batch

In [ ]:
X, Y = get_batch("train")

print("X.shape =", X.shape)
print("Y.shape =", Y.shape)

print()
print("Primeira entrada:")
print(decode(X[0].tolist()))

print()
print("Primeiro alvo:")
print(decode(Y[0].tolist()))

X.shape = torch.Size([32, 64])
Y.shape = torch.Size([32, 64])

Primeira entrada:
 com o gato.
O gato gostava de peixe e o cachorro gostava de car

Primeiro alvo:
com o gato.
O gato gostava de peixe e o cachorro gostava de carn


### Explicação

Criamos um batch de treinamento. Como temos:

**Visão matemática**
B = 32
e
T = 64,

esperamos:

**Visão matemática**
X.shape = (32, 64)
e
Y.shape = (32, 64).

Cada linha representa uma sequência de 64 tokens. A primeira entrada e o primeiro alvo são decodificados novamente para podermos visualizar o que o modelo receberá e o que deverá prever.

# 10. Uma cabeça de atenção

In [ ]:
class Head(nn.Module):
  def __init__(self, d, head_dim, T):
    super().__init__()

    self.key = nn.Linear(
        d,
        head_dim,
        bias=False
    )

    self.query = nn.Linear(
        d,
        head_dim,
        bias=False
    )

    self.value = nn.Linear(
        d,
        head_dim,
        bias=False
    )

    mask = torch.tril(
        torch.ones(T, T)
    )

    self.register_buffer(
        "mask",
        mask
    )

    self.head_dim = head_dim

  def forward(self, x):
    Q = self.query(x)
    K = self.key(x)
    V = self.value(x)

    scores = Q @ K.transpose(-2, -1)

    scores = scores / (self.head_dim ** 0.5)

    T_current = x.size(1)

    mask = self.mask[:T_current, :T_current]

    scores = scores.masked_fill(
        mask == 0,
        float("-inf")
    )

    A = F.softmax(
        scores,
        dim=-1
    )

    out = A @ V

    return out

### Explicação

Esta classe implementa uma cabeça de atenção causal. A entrada possui a forma:

**Visão matemática**
(B, T, d).

A cabeça transforma cada vetor em três representações:

**Visão matemática**
$Q = XW_Q$, $K = XW_K$, $V = XW_V$.

São respectivamente:
* Query;
* Key;
* Value.

Depois calculamos a similaridade entre Queries e Keys:

**Visão matemática**
$QK^\top$.

Dividimos por:

**Visão matemática**
$\sqrt{d_{head}}$

para controlar a escala dos valores. A máscara triangular:

```python
torch.tril(...)
```

impede que uma posição veja posições futuras. Assim, para prever o caractere na posição $t$, o modelo pode utilizar:
$1, \dots, t$,

mas não:
$t + 1, t + 2, \dots$

Depois aplicamos:

**Visão matemática**
$A = \text{softmax}(QK^\top / \sqrt{d_{head}})$

e finalmente:

**Visão matemática**
$\text{out} = AV$.

Essa é a operação central da atenção.

# 11. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d, h, T):
    super().__init__()

    assert d % h == 0

    head_dim = d // h

    self.heads = nn.ModuleList([
        Head(
            d,
            head_dim,
            T
        )
        for _ in range(h)
    ])

    self.projection = nn.Linear(
        d,
        d
    )

  def forward(self, x):
    outputs = [
        head(x)
        for head in self.heads
    ]

    out = torch.cat(
        outputs,
        dim=-1
    )

    out = self.projection(out)

    return out

### Explicação

Uma única cabeça de atenção é multiplicada por:

**Visão matemática**
$h = 4$.

Como:

**Visão matemática**
$d = 32$,

cada cabeça trabalha com:

**Visão matemática**
$d_{head} = \frac{32}{4} = 8$.

Portanto, temos quatro cabeças, cada uma produzindo vetores de dimensão 8. Concatenando:

**Visão matemática**
$8 + 8 + 8 + 8 = 32$.

Por isso, depois da concatenação, voltamos à dimensão:

**Visão matemática**
$d = 32$.

A projeção final permite combinar as informações produzidas pelas quatro cabeças.

# 12. Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, d):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(d, 4 * d),
        nn.GELU(),
        nn.Linear(4 * d, d)
    )

  def forward(self, x):
    return self.net(x)

### Explicação

Depois da atenção, cada posição passa por uma pequena rede neural. Primeiro:

**Visão matemática**
$32 \rightarrow 128$,

pois:

**Visão matemática**
$4d = 4(32) = 128$.

Depois aplicamos a função de ativação GELU. Finalmente:

**Visão matemática**
$128 \rightarrow 32$.

Portanto, a dimensão externa continua sendo $d = 32$. A atenção mistura informações entre posições. A Feed-Forward Network transforma cada posição individualmente.

# 13. Bloco Transformer

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, d, h, T):
    super().__init__()

    self.ln1 = nn.LayerNorm(d)

    self.attention = MultiHeadAttention(
        d,
        h,
        T
    )

    self.ln2 = nn.LayerNorm(d)

    self.ffn = FeedForward(d)

  def forward(self, x):
    x = x + self.attention(
        self.ln1(x)
    )

    x = x + self.ffn(
        self.ln2(x)
    )

    return x

### Explicação

Aqui juntamos os dois componentes principais do Transformer:
1. atenção;
2. rede Feed-Forward.

O bloco possui duas conexões residuais. Primeiro:

**Visão matemática**
$x \rightarrow \text{LayerNorm}(x) \rightarrow \text{Attention} \rightarrow x + \text{Attention}$.

Depois:

**Visão matemática**
$x \rightarrow \text{LayerNorm}(x) \rightarrow \text{FFN} \rightarrow x + \text{FFN}$.

As somas são as conexões residuais.

# 14. Construindo o MiniGPT

In [ ]:
class MiniGPT(nn.Module):
  def __init__(
      self,
      vocab_size,
      d,
      T,
      L,
      h
  ):
    super().__init__()

    self.T = T

    self.token_embedding = nn.Embedding(
        vocab_size,
        d
    )

    self.position_embedding = nn.Embedding(
        T,
        d
    )

    self.blocks = nn.ModuleList([
        TransformerBlock(
            d,
            h,
            T
        )
        for _ in range(L)
    ])

    self.ln_final = nn.LayerNorm(d)

    self.lm_head = nn.Linear(
        d,
        vocab_size
    )

  def forward(self, idx):
    B, T_current = idx.shape

    token_emb = self.token_embedding(idx)

    positions = torch.arange(
        T_current,
        device=idx.device
    )

    position_emb = self.position_embedding(
        positions
    )

    x = token_emb + position_emb

    for block in self.blocks:
      x = block(x)

    x = self.ln_final(x)

    logits = self.lm_head(x)

    return logits

### Explicação

Esta é a arquitetura completa do MiniGPT. Primeiro temos o token embedding:

**Visão matemática**
$\text{token} \rightarrow \mathbb{R}^{32}$.

Depois temos o position embedding, que informa ao modelo a posição de cada token. As duas informações são somadas:

**Visão matemática**
$x = \text{token embedding} + \text{position embedding}$.

Em seguida, $x$ passa pelos:

**Visão matemática**
$L = 2$

blocos Transformer. Depois aplicamos uma última normalização:

```python
self.ln_final
```

e finalmente uma camada linear:

```python
self.lm_head
```

que transforma:

**Visão matemática**
$\mathbb{R}^{32} \rightarrow \mathbb{R}^{|V|}$.

Portanto, para cada posição, o modelo produz um vetor de `vocab_size` números chamados logits. Esses logits representam as pontuações atribuídas pelo modelo aos possíveis próximos caracteres.

# 15. Instanciando o modelo

In [ ]:
model = MiniGPT(
    vocab_size=vocab_size,
    d=d,
    T=T,
    L=L,
    h=h
).to(device)

print(model)

num_paremeters = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Número de parâmetros: {num_paremeters:,}")

MiniGPT(
  (token_embedding): Embedding(36, 32)
  (position_embedding): Embedding(64, 32)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=32, out_features=8, bias=False)
            (query): Linear(in_features=32, out_features=8, bias=False)
            (value): Linear(in_features=32, out_features=8, bias=False)
          )
        )
        (projection): Linear(in_features=32, out_features=32, bias=True)
      )
      (ln2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=32, out_features=128, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=128, out_features=32, bias=True)
        )
      )
    )
  )
  (ln_final): LayerNorm((32,), eps=1e-05, elementwise_

### Explicação

Agora criamos uma instância concreta da arquitetura MiniGPT. O modelo recebe os hiperparâmetros definidos anteriormente. O:

```python
.to(device)
```

move o modelo para CPU ou GPU. Depois contamos todos os parâmetros treináveis:

```python
p.numel()
```

e somamos esses valores. O resultado indica quantos números o treinamento poderá modificar.

# 16. Primeiro forward pass

In [ ]:
X, Y = get_batch("train")

X = X.to(device)
Y = Y.to(device)

logits = model(X)

print("X:       ", X.shape)
print("logits:  ", logits.shape)
print("Y:       ", Y.shape)

X:        torch.Size([32, 64])
logits:   torch.Size([32, 64, 36])
Y:        torch.Size([32, 64])


### Explicação

Agora enviamos um batch para dentro do modelo. Temos:

**Visão matemática**
$X \in \mathbb{N}^{B \times T}$.

Com:

**Visão matemática**
$B = 32, T = 64$.

Portanto:

**Visão matemática**
$X.\text{shape} = (32, 64)$.

O modelo produz:

**Visão matemática**
$\text{logits} \in \mathbb{R}^{32 \times 64 \times |V|}$.

Isso significa:
* 32 sequências;
* 64 posições por sequência;
* uma previsão para cada posição;
* `vocab_size` possíveis caracteres para cada previsão.

# 17. Calculando a loss inicial

In [ ]:
loss_fn = nn.CrossEntropyLoss()

loss = loss_fn(
    logits.reshape(-1, vocab_size),
    Y.reshape(-1)
)

print("Loss inicial:", loss.item())

Loss inicial: 3.7401888370513916


### Explicação

Precisamos medir quão boas são as previsões. Usamos:

`CrossEntropyLoss`

A saída original possui:

**Visão matemática**
$(B, T, |V|)$.

Transformamos isso em:

**Visão matemática**
$(BT, |V|)$.

Da mesma maneira, os alvos passam de:

**Visão matemática**
$(B, T)$

para:

**Visão matemática**
$(BT)$.

Assim, cada uma das $B \times T$ posições é tratada como uma previsão independente.
A loss mede a diferença entre:


**distribuição prevista**
e
**caractere correto**

# 18. Otimizador

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

max_steps = 3000

### Explicação

Aqui definimos o algoritmo que atualizará os parâmetros do modelo. Usamos:

`AdamW`

com taxa de aprendizado:

**Visão matemática**
$\eta = 3 \times 10^{-4}$.

Também definimos:

**Visão matemática**
$3000$

passos de treinamento.

# 19. Treinamento

In [ ]:
for step in range(max_steps):
  X, Y = get_batch("train")

  X = X.to(device)
  Y = Y.to(device)

  logits = model(X)

  loss = loss_fn(
      logits.reshape(-1, vocab_size),
      Y.reshape(-1)
  )

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if step % 200 == 0:
    print(
        f"step {step:4d} | "
        f"loss {loss.item():.4f}"
    )

step    0 | loss 3.7296
step  200 | loss 2.3366
step  400 | loss 1.8992
step  600 | loss 1.7008
step  800 | loss 1.4756
step 1000 | loss 1.2573
step 1200 | loss 1.0157
step 1400 | loss 0.7332
step 1600 | loss 0.5603
step 1800 | loss 0.4403
step 2000 | loss 0.3208
step 2200 | loss 0.2473
step 2400 | loss 0.1814
step 2600 | loss 0.1616
step 2800 | loss 0.1304


### Explicação

Esta é a etapa em que o modelo efetivamente aprende. A cada passo:

1. Criamos um novo batch
```python
X, Y = get_batch("train")
```

2. Fazemos a previsão
```python
logits = model(X)
```

3. Calculamos a loss
```python
loss = loss_fn(...)
```

4. Zeramos os gradientes anteriores
```python
optimizer.zero_grad()
```

5. Calculamos os gradientes
```python
loss.backward()
```

6. Atualizamos os parâmetros
```python
optimizer.step()
```

Portanto, o ciclo fundamental é:

**Conceito-chave**:
X → modelo → logits → loss → gradientes → atualização

Esse processo é repetido 3000 vezes.

# 20. Função de geração

In [30]:
@torch.no_grad()
def generate(
    model,
    idx,
    max_new_tokens
):
    model.eval()

    for _ in range(max_new_tokens):
      # Mantém somente os últimos T tokens
      idx_cond = idx[:, -model.T:]

      # Predições para todas as posições
      logits = model(idx_cond)

      # Queremos somente a última posição
      logits = logits[:, -1, :]

      # Converte logits em probabilidades
      probs = F.softmax(
          logits,
          dim=-1
      )

      # Sorteia o próximo token
      next_token = torch.multinomial(
          probs,
          num_samples=1
      )

      # Adiciona o novo token à sequência
      idx = torch.cat(
          [idx, next_token],
          dim=1
      )

    model.train()

    return idx

### Explicação

Agora criamos a função responsável por gerar texto. Suponha que o prompt seja:

`O g`

O modelo recebe essa sequência e calcula:

**Visão matemática**
$P(x_{t+1} | x_1, \dots, x_t)$.

Pegamos somente os logits da última posição:

```python
logits[:, -1, :]
```

e transformamos esses logits em probabilidades:

**Visão matemática**
$p_i = \text{softmax}(z_i)$.

Depois sorteamos um caractere de acordo com essas probabilidades:

```python
torch.multinomial(...)
```

Esse novo caractere é acrescentado à sequência. O processo se repete. Assim:

**Visão matemática**
$\text{prompt} \rightarrow x_{t+1} \rightarrow x_{t+2} \rightarrow x_{t+3} \rightarrow \dots$

até produzirmos a quantidade solicitada de novos tokens.

# 21. Investigando a próxima previsão

In [31]:
prompt = "ca"

idx = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    logits = model(idx)

    # Pegamos somente a última posição:
    # é ela que prevê o próximo caractere.
    next_logits = logits[:, -1, :]

    probs = F.softmax(
        next_logits,
        dim=-1
    )

probs = probs[0]

top_probs, top_ids = torch.topk(
    probs,
    k=10
)

for p, token_id in zip(top_probs, top_ids):
    ch = itos[token_id.item()]

    print(
        repr(ch),
        f"{p.item():.4f}"
    )


'c' 0.4368
's' 0.2777
'r' 0.1396
'm' 0.0515
'.' 0.0344
'j' 0.0179
'í' 0.0059
't' 0.0056
'O' 0.0050
'ç' 0.0046


### Explicação

Aqui não geramos ainda uma sequência inteira. Queremos observar o que o modelo acha que deve vir depois de “ca”. O prompt:

`ca`

é convertido para tokens e enviado ao modelo. Como temos:

```python
logits[:, -1, :]
```

pegamos somente a previsão correspondente à última posição. Depois transformamos os logits em probabilidades. Finalmente:

```python
torch.topk(probs, k=10)
```

seleciona os dez caracteres com maior probabilidade. Podemos, por exemplo, obter algo conceitualmente semelhante a:

**Saída**
`'t' 0.xxxx`
`'r' 0.xxxx`
`...`

O ponto importante é que o modelo produz uma distribuição de probabilidade sobre o vocabulário.

# 22. Geração de texto

In [39]:
prompt = "O g"

idx = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated = generate(
    model,
    idx,
    max_new_tokens=300
)

result = decode(
    generated[0].tolist()
)

print(result)

O gato gostava de caminhar pelo jardim.
Todias os oservavas os parediarnelte.
Era.
No oo diae, o o do s dim.
ltavava negudo cate.


Ero uma umivez um uma g gato pe gato comevava gato qumpeo de s viniarquega cava ca chorro pelo jardimia ela o perva ela o o carrro go gostavava demia peia e o casaíinho ca


### Explicação

Finalmente utilizamos o modelo como um gerador de texto. O prompt inicial é:

`O g`

O modelo prevê um caractere. Depois utiliza esse novo caractere junto com o contexto anterior para prever o seguinte. Isso continua por:

**300**

novos caracteres. Ao final, convertemos os índices numéricos novamente para caracteres:

```python
decode(...)
```

e imprimimos o texto produzido.

# 23. Visão geral da arquitetura

Temos a seguinte cadeia:

**Visão matemática**

```text
texto
  |
  v
caracteres -> índices
  |
  v
X ∈ ℕ^(B×T)
  |
  v
Token Embedding + Position Embedding
  |
  v
Transformer Block
  |
  v
Transformer Block
  |
  v
LayerNorm
  |
  v
Linear
  |
  v
logits ∈ ℝ^(B×T×|V|)
  |
  v
CrossEntropyLoss
```

Durante o treinamento:

**Conceito-chave**
```text
loss -> backpropagation -> AdamW -> novos parâmetros
```

E, durante a geração:

**Conceito-chave**
```text
prompt -> logits -> softmax -> amostragem -> novo token
novo token -> novo contexto -> próximo token -> ...
```

O resultado é um MiniGPT completo, embora deliberadamente pequeno: ele usa caracteres como tokens, $d = 32$, apenas $L = 2$ blocos Transformer e $h = 4$ cabeças de atenção.